# 02zz - Re-baseline con cleanup dinamico

Re-entrena el modelo CatBoost final del TFG con el cleanup dinamico (en lugar del hardcoded) para establecer el **nuevo baseline honesto** del estudio de validacion.

## Lo que hace este notebook

1. Carga el master base del TFG (`master_table_cutoff.parquet`, 25,200 x 115 cols).
2. Aplica el cleanup dinamico via `scripts/cleanup_pipeline.py` para obtener v1/v2/v3 nuevos.
3. Compara la composicion del v3 dinamico vs v3 hardcoded del TFG.
4. Re-entrena CatBoost final (con best_params de Optuna del TFG) sobre el master v3 dinamico, usando los mismos splits del TFG (`splits_indices_cutoff.parquet`).
5. Aplica calibracion isotonic para churn_30d (uncalibrated para churn_14d).
6. Reporta AUC test de cada target y compara con los AUC originales del TFG (0.847 / 0.795).

## El nuevo baseline a batir

Tras este notebook, el AUC con cleanup dinamico es el **nuevo baseline** para el estudio. Las 54 configs se comparan contra este, no contra los AUC originales.


In [1]:
# [SETUP]
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[0]))

import pandas as pd
import numpy as np
import json, time
from datetime import datetime

from scripts import config as cfg
from scripts import cleanup_pipeline as cp

from catboost import CatBoostClassifier, Pool
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss, f1_score

np.random.seed(cfg.RANDOM_SEED)


In [2]:
# [EXEC] 1. Cargar master base del TFG
master_path = cfg.TFG_DATA_QC / "master_table_cutoff.parquet"
master = pd.read_parquet(master_path)
print(f'Master base cargado: {master.shape}')
print(f'Targets: {[c for c in master.columns if "churn_" in c]}')


Master base cargado: (25200, 115)
Targets: ['churn_14d', 'churn_30d']


In [3]:
# [EXEC] 2. Aplicar cleanup dinamico
print('Aplicando cleanup dinamico...')
t0 = time.time()
drops = cp.compute_dynamic_cleanup(master, target_cols=['churn_14d','churn_30d'])
print(f'  high_missing:    {len(drops["high_missing"])} cols')
print(f'  quasi_constant:  {len(drops["quasi_constant"])} cols')
print(f'  target_leakage:  {len(drops["target_leakage"])} cols')
print(f'  corr_99_drop:    {len(drops["corr_99_drop"])} cols')
print(f'  corr_95_drop:    {len(drops["corr_95_drop"])} cols')

masters_dynamic = cp.apply_cleanups(master, drops)
print(f'\nv1: {masters_dynamic["v1_conservative"].shape}')
print(f'v2: {masters_dynamic["v2_intermediate"].shape}')
print(f'v3: {masters_dynamic["v3_aggressive"].shape}')
print(f'\nCleanup time: {time.time()-t0:.1f}s')


Aplicando cleanup dinamico...
  high_missing:    2 cols
  quasi_constant:  32 cols
  target_leakage:  13 cols
  corr_99_drop:    6 cols
  corr_95_drop:    6 cols

v1: (25200, 68)
v2: (25200, 62)
v3: (25200, 56)

Cleanup time: 0.2s


In [4]:
# [ANALYSIS] 3. Comparar con master v3 hardcoded del TFG
master_v3_tfg = pd.read_parquet(cfg.TFG_DATA_QC / "master_table_cutoff_v3_aggressive.parquet")
master_v3_dyn = masters_dynamic['v3_aggressive']

cols_tfg = set(master_v3_tfg.columns)
cols_dyn = set(master_v3_dyn.columns)

print(f'TFG v3 (hardcoded): {len(cols_tfg)} cols')
print(f'Dynamic v3:         {len(cols_dyn)} cols')
print(f'\nCols solo en TFG (sobrevivieron al hardcoded pero NO al dynamic):')
for c in sorted(cols_tfg - cols_dyn):
    print(f'  - {c}')
print(f'\nCols solo en dynamic (mantenidas por dynamic, eliminadas por hardcoded):')
for c in sorted(cols_dyn - cols_tfg):
    print(f'  + {c}')


TFG v3 (hardcoded): 80 cols
Dynamic v3:         56 cols

Cols solo en TFG (sobrevivieron al hardcoded pero NO al dynamic):
  - char_attack_max
  - device_is_multi_platform
  - feedback_has_any
  - feedback_n_monetization
  - feedback_n_negative
  - feedback_n_positive
  - feedback_total
  - iap_consumables_count
  - iap_consumables_unique_products
  - iap_has_annual
  - iap_has_consumables
  - iap_has_quarterly
  - iap_has_subscription_ever
  - iap_is_current_payer
  - iap_is_payer
  - iap_is_subscription_active
  - iap_paid_last_30d
  - iap_paid_last_90d
  - iap_subscription_active_last_7d
  - iap_subscriptions_count
  - iap_trial_only
  - items_mean_defense
  - reward_has_premium
  - reward_last_claim_days_ago
  - reward_premium_days_max
  - reward_records_total
  - user_created_at_days_ago

Cols solo en dynamic (mantenidas por dynamic, eliminadas por hardcoded):
  + char_attack_defense_sum_max
  + items_unique_definitions
  + user_player_lifespan_days


In [5]:
# [EXEC] 4. Guardar masters re-bauliados (locales al estudio)
out_dir = cfg.STUDY_MASTERS / 'rebaseline'
out_dir.mkdir(parents=True, exist_ok=True)

for version, df in masters_dynamic.items():
    out = out_dir / f'master_rebaseline_{version}.parquet'
    df.to_parquet(out, compression='snappy', index=False)
    print(f'Guardado: {out.name} ({df.shape})')

# Drops report
report_df = cp.generate_drops_report(drops, out_dir / 'dropped_cols.csv')
print(f'\nDrops report: {len(report_df)} filas')


Guardado: master_rebaseline_v1_conservative.parquet ((25200, 68))
Guardado: master_rebaseline_v2_intermediate.parquet ((25200, 62))
Guardado: master_rebaseline_v3_aggressive.parquet ((25200, 56))

Drops report: 59 filas


In [6]:
# [EXEC] 5. Preparar features + splits (replica 03z_final_model)
master_use = masters_dynamic['v3_aggressive']
splits_df = pd.read_parquet(cfg.TFG_DATA_MODELS / 'splits_indices_cutoff.parquet')

# Alinear splits con master por user_id
if 'user_id' in splits_df.columns:
    splits_aligned = splits_df.set_index('user_id').reindex(master_use['user_id'].values).reset_index()
else:
    splits_aligned = splits_df

# Cast categoricas
cat_cols = master_use.select_dtypes(include=['object','string','category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'user_id']
for c in cat_cols:
    master_use[c] = master_use[c].astype(str).fillna('missing').replace('nan','missing')

X_full = master_use.drop(columns=['churn_14d','churn_30d','user_id'])
trainval_mask = splits_aligned['split'].values != 'test'
val_mask = splits_aligned['split'].values == 'val'
test_mask = splits_aligned['split'].values == 'test'

X_trainval = X_full[trainval_mask].reset_index(drop=True)
X_val = X_full[val_mask].reset_index(drop=True)
X_test = X_full[test_mask].reset_index(drop=True)
print(f'X_trainval={X_trainval.shape} | X_val={X_val.shape} | X_test={X_test.shape}')
print(f'cat_cols ({len(cat_cols)}): {cat_cols}')


X_trainval=(21420, 53) | X_val=(3781, 53) | X_test=(3780, 53)
cat_cols (5): ['device_primary_platform', 'country', 'has_user_rated_app', 'user_created_at', 'user_store_where_published']


In [7]:
# [EXEC] 6. Cargar best_params del TFG (Optuna)
TUNING_DIR = cfg.TFG_INFORMES / 'fase3_modeling' / '03g_tuning_catboost'
best_params = {}
for target in ['churn_14d', 'churn_30d']:
    with open(TUNING_DIR / f'best_params_{target}.json') as f:
        best_params[target] = json.load(f)
    print(f'best_params[{target}]: {best_params[target]}')


best_params[churn_14d]: {'iterations': 590, 'learning_rate': 0.04831657963496582, 'depth': 4, 'l2_leaf_reg': 5.641017040821281, 'border_count': 47, 'bagging_temperature': 0.24554232102189724, 'random_strength': 7.477893566373788, 'min_data_in_leaf': 6}
best_params[churn_30d]: {'iterations': 1218, 'learning_rate': 0.01644266940285884, 'depth': 7, 'l2_leaf_reg': 1.7408592532406044, 'border_count': 55, 'bagging_temperature': 0.005258618458008513, 'random_strength': 0.6822584674450662, 'min_data_in_leaf': 45}


In [8]:
# [EXEC] 7. Entrenar + calibrar + evaluar sobre TEST
final_metrics = []
for target in ['churn_14d', 'churn_30d']:
    print(f'\n=== {target} ===')
    y = master_use[target].astype(int)
    y_trainval = y[trainval_mask]
    y_val = y[val_mask]
    y_test = y[test_mask]

    params = {**best_params[target], 'random_state': cfg.RANDOM_SEED,
              'eval_metric':'AUC', 'verbose':False, 'thread_count':-1}
    train_pool = Pool(X_trainval, y_trainval, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)
    test_pool = Pool(X_test, y_test, cat_features=cat_cols)

    model = CatBoostClassifier(**params)
    model.fit(train_pool, verbose=False)

    proba_test_raw = model.predict_proba(test_pool)[:,1]
    proba_val_raw = model.predict_proba(val_pool)[:,1]

    # churn_30d -> isotonic, churn_14d -> uncalibrated (best_method del TFG)
    if target == 'churn_30d':
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(proba_val_raw, y_val)
        proba_test_cal = cal.predict(proba_test_raw)
        cal_method = 'isotonic'
    else:
        proba_test_cal = proba_test_raw
        cal_method = 'uncalibrated'

    pred = (proba_test_cal >= 0.5).astype(int)
    auc = float(roc_auc_score(y_test, proba_test_cal))
    auc_pr = float(average_precision_score(y_test, proba_test_cal))
    brier = float(brier_score_loss(y_test, proba_test_cal))
    f1 = float(f1_score(y_test, pred))

    auc_orig = cfg.BASELINE_CONFIG[f'reference_auc_{target.split("_")[1]}_original']
    delta = auc - auc_orig

    final_metrics.append({
        'target': target, 'calibration': cal_method,
        'auc_roc_rebaseline': auc, 'auc_pr_rebaseline': auc_pr,
        'brier_rebaseline': brier, 'f1_rebaseline': f1,
        'auc_roc_original_tfg': auc_orig, 'delta_auc': delta,
        'n_features_used': X_full.shape[1],
    })
    print(f'  AUC ({cal_method}): {auc:.4f} | Original TFG: {auc_orig:.4f} | Delta: {delta:+.4f}')
    print(f'  AUC-PR: {auc_pr:.4f} | Brier: {brier:.4f} | F1: {f1:.4f}')

metrics_df = pd.DataFrame(final_metrics)
print(f'\n=== Final metrics ===')
print(metrics_df.round(4).to_string(index=False))



=== churn_14d ===
  AUC (uncalibrated): 0.8453 | Original TFG: 0.8470 | Delta: -0.0017
  AUC-PR: 0.8079 | Brier: 0.1317 | F1: 0.6957

=== churn_30d ===
  AUC (isotonic): 0.7912 | Original TFG: 0.7950 | Delta: -0.0038
  AUC-PR: 0.8019 | Brier: 0.1864 | F1: 0.6925

=== Final metrics ===
   target  calibration  auc_roc_rebaseline  auc_pr_rebaseline  brier_rebaseline  f1_rebaseline  auc_roc_original_tfg  delta_auc  n_features_used
churn_14d uncalibrated              0.8453             0.8079            0.1317         0.6957                 0.847    -0.0017               53
churn_30d     isotonic              0.7912             0.8019            0.1864         0.6925                 0.795    -0.0038               53


In [9]:
# [REPORT] 02zz_rebaseline_report.md
report = f"""# Re-baseline con cleanup dinamico - 02zz

**Fecha**: {datetime.now()}
**Sample**: master_table_cutoff.parquet del TFG (N={len(master):,}, 115 cols base)
**Splits**: heredados del TFG (splits_indices_cutoff.parquet, 70/15/15 estratificado)
**Hyperparametros**: best_params Optuna del TFG (03g_tuning_catboost)
**Calibracion**: isotonic para churn_30d, uncalibrated para churn_14d (consistente con TFG)

## Comparacion v3 hardcoded vs v3 dinamico

| Metrica | TFG v3 (hardcoded) | Dynamic v3 |
|---|---:|---:|
| N_columnas | {len(cols_tfg)} | {len(cols_dyn)} |
| Cols dropeadas dinamicamente que sobrevivian en TFG | - | {len(cols_tfg - cols_dyn)} |
| Cols mantenidas dinamicamente que eliminaba TFG | - | {len(cols_dyn - cols_tfg)} |

### Cols eliminadas por dynamic pero presentes en TFG v3
{chr(10).join(f"- {c}" for c in sorted(cols_tfg - cols_dyn))}

### Cols presentes en dynamic v3 pero ausentes en TFG v3
{chr(10).join(f"+ {c}" for c in sorted(cols_dyn - cols_tfg))}

## Drops dinamicos por categoria

- high_missing:   {len(drops['high_missing'])} cols
- quasi_constant: {len(drops['quasi_constant'])} cols
- target_leakage: {len(drops['target_leakage'])} cols
- corr_99_drop:   {len(drops['corr_99_drop'])} cols
- corr_95_drop:   {len(drops['corr_95_drop'])} cols

## Resultados AUC sobre TEST

| Target | Calibration | AUC rebaseline | AUC original TFG | Delta |
|---|---|---:|---:|---:|
{chr(10).join(f"| {m['target']} | {m['calibration']} | {m['auc_roc_rebaseline']:.4f} | {m['auc_roc_original_tfg']:.4f} | {m['delta_auc']:+.4f} |" for m in final_metrics)}

## Interpretacion

Los AUC con cleanup dinamico son el **nuevo baseline a batir** en el estudio de validacion. Las 54 configs del grid se compararan contra estos valores, no contra los AUC originales del TFG.

Si delta > 0: el cleanup dinamico es estrictamente mejor (sin sorpresa, elimina redundancias que el hardcoded dejaba pasar).
Si delta < 0: el cleanup dinamico es ligeramente peor (improbable; significaria que features que el hardcoded conservaba aportan informacion neta).
Si delta ~ 0: el cleanup hardcoded ya era casi-optimo; el rebaseline confirma su validez.

## Outputs

- `data/masters/rebaseline/master_rebaseline_v{{1,2,3}}.parquet`
- `data/masters/rebaseline/dropped_cols.csv`
- `informes/02zz_rebaseline_report.md` (este archivo)
"""

(cfg.STUDY_INFORMES / '02zz_rebaseline_report.md').write_text(report)

# Tambien guardar metrics_df
metrics_df.to_csv(cfg.STUDY_INFORMES / '02zz_rebaseline_metrics.csv', index=False)

print(f'\n[OK] Reporte: {cfg.STUDY_INFORMES / "02zz_rebaseline_report.md"}')
print(f'[OK] Metrics CSV: {cfg.STUDY_INFORMES / "02zz_rebaseline_metrics.csv"}')



[OK] Reporte: /Users/jezquerro/Documents/tfg/validation_study/informes/02zz_rebaseline_report.md
[OK] Metrics CSV: /Users/jezquerro/Documents/tfg/validation_study/informes/02zz_rebaseline_metrics.csv


In [10]:
# [EXEC] Actualizar BASELINE_CONFIG con los nuevos AUC del rebaseline
# (Solo para informar al usuario; el config.py se actualiza manualmente despues)
print('Sugerencia: actualizar BASELINE_CONFIG en scripts/config.py con:')
for m in final_metrics:
    win = m['target'].split('_')[1]
    print(f"  reference_auc_{win}_rebaseline = {m['auc_roc_rebaseline']:.4f}")


Sugerencia: actualizar BASELINE_CONFIG en scripts/config.py con:
  reference_auc_14d_rebaseline = 0.8453
  reference_auc_30d_rebaseline = 0.7912
